# Build circuits in C, simulate in Python

## Prerequisites

* Python 3.10+ with this branch's built `qdk`, `ipykernel`, and IPython in the selected Python kernel, plus JupyterLab or VS Code's Python/Jupyter extensions.
* A checkout of this branch, Rust/Cargo, and a GCC/Clang-compatible C compiler (`CC` may select it).
* The branch-matched native circuit library and header. From the repository root, run `python source/circuit_capi/build.py` after completing the [setup guide](README.md).
* macOS or Linux; use WSL2 for this native sample on Windows.
* Optional GPU execution requires a QDK-supported GPU and drivers. It is disabled by default.

This is a Python-kernel notebook, not a C-kernel notebook. It displays, compiles, and executes the standalone [C client](native/circuits.c), then loads its binary snapshots in Python. Native circuit construction does not load Python. Neither circuit API has a direct simulation method; simulation uses QIR and `qdk.simulation.run_qir`.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import math
import sys
from IPython.display import Markdown, display
from qdk import Result, TargetProfile
from qdk.circuit import QuantumCircuit
from qdk.simulation import run_qir

root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "source/circuit_capi/includes/qdk_circuit.h").is_file()
    ),
    None,
)
if root is None:
    raise RuntimeError("Start this kernel inside the QDK checkout.")
sys.path.insert(0, str(root / "samples/notebooks/circuit_api"))
from _native import SOURCES, build_and_run

RUN_GPU = False
SHOTS = 128

## Build Base, Adaptive, and symbolic circuits in C

Read `main` in the source displayed below:

1. Append H and CX, then measure both qubits to build a Base-profile Bell circuit.
2. Measure qubit 0, build a clbit expression, and open an if-body that applies X to qubit 1. This runtime feedback requires Adaptive QIR.
3. Create an unbound Rx parameter named `theta`, ready for Python to sweep later.

`QdkOperation` descriptors borrow their arrays for the call. Builders, expressions, and parameters are dependent handles: finish builders and release dependents before freeing the circuit. Exported `QdkBytes` buffers are owned, length-delimited data, not NUL-terminated strings. Write the exact length and release each buffer with `qdk_bytes_free`.

Edit [circuits.c](native/circuits.c) to experiment, then rerun the compile cell. The C program checks every QDK status and reports diagnostics.

In [ ]:
source = (SOURCES / "circuits.c").read_text(encoding="utf-8")
display(Markdown(f"```c\n{source}\n```"))

In [ ]:
work = TemporaryDirectory(prefix="qdk-c-circuits-")
directory = Path(work.name).resolve()
print("Artifact directory:", directory)
build_and_run("c", directory)
print("Native artifacts (removed by the final cleanup cell):")
for path in sorted(directory.iterdir()):
    print(path)

## Load the native snapshots and run on the CPU

The `.qdk` files contain typed binary snapshots, not JSON or HUGR documents. Each has a readable `.sexpr` snapshot alongside it. Python loads both formats and checks that they restore the same circuit without reconstructing the C operations. Compiling it again produces the same QIR as the native C export. Both examples produce only `00` and `11`: the Bell circuit uses entanglement, while the feedback circuit copies a measured classical bit into a second qubit.

We request `type='cpu'` explicitly. Assertions check shot count and allowed outcomes rather than demanding exactly equal counts.

In [ ]:
programs = {}
for name, profile in [
    ("base", TargetProfile.Base),
    ("adaptive", TargetProfile.Adaptive),
]:
    payload = (directory / f"{name}.qdk").read_bytes()
    circuit = QuantumCircuit.from_bytes(payload)
    assert circuit.to_bytes() == payload
    text = (directory / f"{name}.sexpr").read_text(encoding="utf-8")
    assert QuantumCircuit.from_text(text).to_bytes() == payload
    programs[name] = circuit.to_qir(profile)
    assert programs[name] == (directory / f"{name}.ll").read_text(encoding="utf-8")
    print(name, "\n", circuit)


def correlated_counts(samples):
    assert len(samples) == SHOTS
    assert all(
        len(s) == 2 and s[0] == s[1] and s[0] in (Result.Zero, Result.One)
        for s in samples
    )
    return Counter(
        "".join("1" if bit == Result.One else "0" for bit in s) for s in samples
    )


for name, qir in programs.items():
    samples = run_qir(qir, shots=SHOTS, seed=42, type="cpu")
    print(name, "CPU:", correlated_counts(samples))

## Opt in to GPU simulation

Set `RUN_GPU = True` in the setup cell and rerun this cell after the CPU section. GPU failures are surfaced, never silently redirected to the CPU. Identical seeds do not promise identical samples across backends, and tiny circuits are not a GPU performance benchmark.

In [ ]:
if RUN_GPU:
    for name, qir in programs.items():
        samples = run_qir(qir, shots=SHOTS, seed=42, type="gpu")
        print(name, "GPU:", correlated_counts(samples))
else:
    print("GPU execution disabled; set RUN_GPU = True to enable it.")

## Bind a C-built symbolic template in Python

The C process has already freed its parameter handle, but the snapshot retains the symbol. Obtain its new Python-owned parameter from the loaded circuit. Binding returns a new circuit and leaves the template unbound. Rx(0) always measures Zero; Rx(pi) always measures One.

In [ ]:
template = QuantumCircuit.from_bytes((directory / "symbolic.qdk").read_bytes())
text = (directory / "symbolic.sexpr").read_text(encoding="utf-8")
assert QuantumCircuit.from_text(text).to_bytes() == template.to_bytes()
(theta,) = template.parameters
sweep = {}
for angle in [0.0, math.pi / 2, math.pi]:
    bound = template.assign_parameters([(theta, angle)])
    assert bound.parameters == []
    samples = run_qir(
        bound.to_qir(TargetProfile.Base), shots=SHOTS, seed=42, type="cpu"
    )
    assert len(samples) == SHOTS
    assert all(len(s) == 1 and s[0] in (Result.Zero, Result.One) for s in samples)
    sweep[angle] = samples
    print(f"Rx({angle:.3f}):", Counter(str(s[0]) for s in samples))
assert all(s == [Result.Zero] for s in sweep[0.0])
assert all(s == [Result.One] for s in sweep[math.pi])
assert template.parameters == [theta]

## Cleanup

Copy any `.qdk`, `.sexpr`, or `.ll` files you want to keep out of `directory` before running this cell. The temporary directory contains only this notebook's generated artifacts; the editable source stays in the checkout. See the [C interface guide](../../../source/circuit_capi/README.md) for the full ABI.

In [ ]:
work.cleanup()
assert not directory.exists()